In [28]:
! pip install transformers==4.35.2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [29]:
# translator_lib_hstate.py
# transformers==4.35.2 전제 (DynamicCache 사용 안 함)

from dataclasses import dataclass
from typing import Tuple, List

import torch
import torch.nn as nn


@dataclass
class ModelKVSpec:
    n_layers: int
    n_heads: int
    head_dim: int
    hidden_size: int


def pack_past_key_values(
    past_key_values: Tuple[Tuple[torch.Tensor, torch.Tensor], ...]
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    HF GPT2 past_key_values:
      tuple(n_layers) of (k, v)
      k,v: [B, n_heads, S, head_dim]
    return:
      K,V: [B, S, L, hidden_size]
    """
    keys: List[torch.Tensor] = []
    values: List[torch.Tensor] = []

    for (k, v) in past_key_values:
        k2 = k.permute(0, 2, 1, 3).contiguous().view(k.size(0), k.size(2), -1)
        v2 = v.permute(0, 2, 1, 3).contiguous().view(v.size(0), v.size(2), -1)
        keys.append(k2)
        values.append(v2)

    K = torch.stack(keys, dim=2)   # [B, S, L, D]
    V = torch.stack(values, dim=2) # [B, S, L, D]
    return K, V


def unpack_past_key_values(
    K: torch.Tensor,
    V: torch.Tensor,
    spec: ModelKVSpec
) -> Tuple[Tuple[torch.Tensor, torch.Tensor], ...]:
    """
    K,V: [B, S, L, hidden_size]
    return HF past_key_values tuple:
      k,v: [B, n_heads, S, head_dim]
    """
    B, S, L, D = K.shape
    assert L == spec.n_layers, (L, spec.n_layers)
    assert D == spec.hidden_size, (D, spec.hidden_size)

    out = []
    for layer in range(L):
        k = K[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        v = V[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        out.append((k, v))
    return tuple(out)


@torch.no_grad()
def extract_layer_inputs_gpt2(model, input_ids, attention_mask=None, use_cache=True):
    """
    "진짜 hidden states" = 각 블록 입력 (ln_1 통과 전)만 반환
    return:
      H_raw: [B,S,L,d_model]
      past_key_values
      hidden_states tuple
    """
    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=use_cache,
        output_hidden_states=True,
        return_dict=True,
    )
    hs = out.hidden_states
    L = model.config.n_layer

    H_layers = []
    for l in range(L):
        H_layers.append(hs[l])  # pre-ln_1 raw hidden

    H_raw = torch.stack(H_layers, dim=2)  # [B,S,L,d]
    return H_raw, out.past_key_values, hs


def recompute_kv_from_hidden_raw_gpt2(model, H_raw: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    raw hidden (ln_1 전)에서 KV 복구:
      K^L = ln_1^L(H_raw^L) @ Wk^L + bk^L
      V^L = ln_1^L(H_raw^L) @ Wv^L + bv^L
    return:
      K,V: [B,S,L,d_model]
    """
    B, S, L, D = H_raw.shape
    assert L == model.config.n_layer
    assert D == model.config.n_embd

    Ks, Vs = [], []
    d = D

    for l in range(L):
        block = model.transformer.h[l]
        attn = block.attn

        Hl = block.ln_1(H_raw[:, :, l, :])  # [B,S,d]

        W = attn.c_attn.weight  # [d, 3d]
        b = attn.c_attn.bias    # [3d]

        Wk = W[:, d:2*d]
        Wv = W[:, 2*d:3*d]
        bk = b[d:2*d]
        bv = b[2*d:3*d]

        Kl = Hl @ Wk + bk
        Vl = Hl @ Wv + bv
        Ks.append(Kl)
        Vs.append(Vl)

    K = torch.stack(Ks, dim=2)  # [B,S,L,d]
    V = torch.stack(Vs, dim=2)  # [B,S,L,d]
    return K, V


def cosine_sim_flat(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> float:
    a1 = a.detach().float().reshape(-1)
    b1 = b.detach().float().reshape(-1)
    denom = (a1.norm() * b1.norm()).clamp_min(eps)
    return float(torch.dot(a1, b1) / denom)


class CrossAttnBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, int(d_model * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(d_model * mlp_ratio), d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mem: torch.Tensor) -> torch.Tensor:
        q = self.ln_q(x)
        kv = self.ln_kv(mem)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


class LocalToSharedAdapterH(nn.Module):
    """
    T[mi -> Σ] : H_raw [B,S,L,Di] -> H* [B,S,Q]
    """
    def __init__(self, n_layers: int, d_in: int, q_dim: int, d_model: int = 256, n_heads: int = 8,
                 mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.n_layers = n_layers
        self.d_in = d_in
        self.q_dim = q_dim

        self.in_ln = nn.LayerNorm(d_in)
        self.in_proj = nn.Linear(d_in, d_model)
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        self.out_ln = nn.LayerNorm(n_layers * d_model)
        self.out_proj = nn.Linear(n_layers * d_model, q_dim)
        self.act = nn.GELU()

    def forward(self, H_raw: torch.Tensor) -> torch.Tensor:
        B, S, L, Di = H_raw.shape
        assert L == self.n_layers and Di == self.d_in

        h = self.act(self.in_proj(self.in_ln(H_raw)))  # [B,S,L,d_model]
        cur = h[:, :, 0, :]
        outs = []
        for l, blk in enumerate(self.blocks):
            cur = blk(cur, h[:, :, l, :])
            outs.append(cur)
        cat = torch.cat(outs, dim=-1)
        y = self.act(self.out_proj(self.out_ln(cat)))
        return y  # [B,S,Q]


class SharedToLocalAdapterH(nn.Module):
    """
    T[Σ -> mi] : H* [B,S,Q] -> H_raw [B,S,L,Di]
    """
    def __init__(self, n_layers: int, q_dim: int, d_out: int, d_model: int = 256, n_heads: int = 8,
                 mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        assert q_dim % n_layers == 0, f"q_dim({q_dim}) must be divisible by n_layers({n_layers})"

        self.n_layers = n_layers
        self.q_dim = q_dim
        self.d_out = d_out
        self.q_per_layer = q_dim // n_layers

        self.in_ln = nn.LayerNorm(self.q_per_layer)
        self.in_proj = nn.Linear(self.q_per_layer, d_model)
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        self.out_ln = nn.LayerNorm(n_layers * d_model)
        self.out_proj = nn.Linear(n_layers * d_model, n_layers * d_out)
        self.act = nn.GELU()

    def forward(self, Hs: torch.Tensor) -> torch.Tensor:
        B, S, Q = Hs.shape
        assert Q == self.q_dim

        x = Hs.view(B, S, self.n_layers, self.q_per_layer)
        h = self.act(self.in_proj(self.in_ln(x)))  # [B,S,L,d_model]

        cur = h[:, :, 0, :]
        outs = []
        for l, blk in enumerate(self.blocks):
            cur = blk(cur, h[:, :, l, :])
            outs.append(cur)

        cat = torch.cat(outs, dim=-1)
        y = self.act(self.out_proj(self.out_ln(cat)))
        return y.view(B, S, self.n_layers, self.d_out)


class SharedSpaceHTranslator(nn.Module):
    """
    One-way raw hidden translator:
      B (gpt2-medium) -> Σ -> A (gpt2)

    Loss는 밖에서
      H_B_raw --translator--> H_A_raw_hat
      H_A_raw_hat --(ln_1 + Wk/Wv)--> K_A_hat, V_A_hat
    후에 KV reconstruction loss로 계산
    """
    def __init__(
        self,
        a_layers: int, a_hidden: int,
        b_layers: int, b_hidden: int,
        q_dim: int = 1536,
        d_model: int = 256,
        n_heads: int = 8,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.B_to_S = LocalToSharedAdapterH(b_layers, b_hidden, q_dim, d_model, n_heads, dropout=dropout)
        self.S_to_A = SharedToLocalAdapterH(a_layers, q_dim, a_hidden, d_model, n_heads, dropout=dropout)

    def translate_B_to_A(self, H_B_raw: torch.Tensor) -> torch.Tensor:
        Hs = self.B_to_S(H_B_raw)
        H_A_raw = self.S_to_A(Hs)
        return H_A_raw

In [30]:
# code1_train_hstate_translator.py
# 목표: Raw HiddenStates Translator 학습 + (ln_1 + Wk/Wv)로 KV recompute 후 reconstruction loss
# - 방향: gpt2-medium(B) -> gpt2(A)
# - WikiText 2000 step
# - 매 100 step마다 validation loss 출력
#
# transformers==4.35.2 전제, argparse 금지

import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


MODEL_A_NAME = "gpt2"
MODEL_B_NAME = "gpt2-medium"

SEED = 42
CONTEXT_LEN = 128
BATCH_SIZE = 2
TRAIN_STEPS = 10000
VAL_EVERY = 100
VAL_BATCHES = 10

LR = 1e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0

Q_DIM = 3072
D_MODEL = 512
N_HEADS = 8
DROPOUT = 0.0

SAVE_PATH = "hstate_translator_ckpt.pt"


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_lm_blocks(tokenizer, texts, block_size: int):
    joined = "\n\n".join([t for t in texts if t is not None])
    ids = tokenizer(joined, return_tensors=None)["input_ids"]
    n_blocks = len(ids) // block_size
    ids = ids[: n_blocks * block_size]
    return [ids[i * block_size : (i + 1) * block_size] for i in range(n_blocks)]


def collate_fn(batch):
    input_ids = torch.tensor(batch, dtype=torch.long)
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask}


def recon_loss_via_kv(model_target, H_translated_raw, K_target, V_target):
    K_hat, V_hat = recompute_kv_from_hidden_raw_gpt2(model_target, H_translated_raw)
    return torch.mean((K_hat - K_target) ** 2) + torch.mean((V_hat - V_target) ** 2)


@torch.no_grad()
def eval_loss(translator, modelA, modelB, val_loader, device):
    translator.eval()
    total, n = 0.0, 0

    for i, batch in enumerate(val_loader):
        if i >= VAL_BATCHES:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        H_A_raw, pastA, _ = extract_layer_inputs_gpt2(modelA, input_ids, attention_mask, use_cache=True)
        H_B_raw, _, _ = extract_layer_inputs_gpt2(modelB, input_ids, attention_mask, use_cache=True)

        K_A, V_A = pack_past_key_values(pastA)

        H_B_to_A_raw = translator.translate_B_to_A(H_B_raw)
        loss = recon_loss_via_kv(modelA, H_B_to_A_raw, K_A, V_A)

        total += float(loss.item())
        n += 1

    translator.train()
    return total / max(1, n)


def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_A_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    ds = load_dataset("wikitext", "wikitext-2-raw-v1")
    train_blocks = make_lm_blocks(tokenizer, ds["train"]["text"], CONTEXT_LEN)
    val_blocks = make_lm_blocks(tokenizer, ds["validation"]["text"], CONTEXT_LEN)

    train_loader = DataLoader(train_blocks, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_blocks, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=False)

    modelA = GPT2LMHeadModel.from_pretrained(MODEL_A_NAME).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(MODEL_B_NAME).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    for p in modelA.parameters():
        p.requires_grad_(False)
    for p in modelB.parameters():
        p.requires_grad_(False)

    translator = SharedSpaceHTranslator(
        a_layers=modelA.config.n_layer, a_hidden=modelA.config.n_embd,
        b_layers=modelB.config.n_layer, b_hidden=modelB.config.n_embd,
        q_dim=Q_DIM, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT
    ).to(device).train()

    opt = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_iter = iter(train_loader)

    for step in range(1, TRAIN_STEPS + 1):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            H_A_raw, pastA, _ = extract_layer_inputs_gpt2(modelA, input_ids, attention_mask, use_cache=True)
            H_B_raw, _, _ = extract_layer_inputs_gpt2(modelB, input_ids, attention_mask, use_cache=True)
            K_A, V_A = pack_past_key_values(pastA)

        opt.zero_grad(set_to_none=True)

        H_B_to_A_raw = translator.translate_B_to_A(H_B_raw)
        loss = recon_loss_via_kv(modelA, H_B_to_A_raw, K_A, V_A)

        loss.backward()
        nn.utils.clip_grad_norm_(translator.parameters(), GRAD_CLIP_NORM)
        opt.step()

        if step % 10 == 0:
            print(f"[train] step={step:4d} loss={loss.item():.6f}")

        if step % VAL_EVERY == 0:
            val_loss = eval_loss(translator, modelA, modelB, val_loader, device)
            print(f"[valid] step={step:4d} recon_loss={val_loss:.6f}")

    ckpt = {
        "translator_state": translator.state_dict(),
        "config": {
            "MODEL_A_NAME": MODEL_A_NAME,
            "MODEL_B_NAME": MODEL_B_NAME,
            "Q_DIM": Q_DIM,
            "D_MODEL": D_MODEL,
            "N_HEADS": N_HEADS,
            "DROPOUT": DROPOUT,
            "a_layers": modelA.config.n_layer,
            "a_hidden": modelA.config.n_embd,
            "b_layers": modelB.config.n_layer,
            "b_hidden": modelB.config.n_embd,
        },
    }
    torch.save(ckpt, SAVE_PATH)
    print("saved:", SAVE_PATH)


if __name__ == "__main__":
    main()

device: cuda


Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors


[train] step=  10 loss=4.156606
[train] step=  20 loss=3.806466
[train] step=  30 loss=3.627525
[train] step=  40 loss=3.585621
[train] step=  50 loss=3.515000
[train] step=  60 loss=3.502848
[train] step=  70 loss=3.401818
[train] step=  80 loss=3.316471
[train] step=  90 loss=3.333534
[train] step= 100 loss=3.262095
[valid] step= 100 recon_loss=3.290312
[train] step= 110 loss=3.328127
[train] step= 120 loss=3.249007
[train] step= 130 loss=3.232601
[train] step= 140 loss=3.220426
[train] step= 150 loss=3.121724
[train] step= 160 loss=3.078434
[train] step= 170 loss=3.239680
[train] step= 180 loss=2.854182
[train] step= 190 loss=2.758672
[train] step= 200 loss=2.518856
[valid] step= 200 recon_loss=2.576846
[train] step= 210 loss=2.508529
[train] step= 220 loss=2.390171
[train] step= 230 loss=2.406374
[train] step= 240 loss=2.321787
[train] step= 250 loss=2.344463
[train] step= 260 loss=2.159915
[train] step= 270 loss=2.281797
[train] step= 280 loss=2.220794
[train] step= 290 loss=2.101

In [31]:
# code2_infer_hstate_translate.py
# 목표:
# - 학습된 Raw HiddenStates Translator 기반: gpt2-medium(B) -> gpt2(A)
# - raw hidden 번역 후 (ln_1 + Wk/Wv)로 KV recompute
# - Prefix 2개에 대해:
#     1) A_original  -> model A
#     2) B_to_A      -> model A
#     3) B_original  -> model B
#
# transformers==4.35.2 전제, argparse 금지

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


CKPT_PATH = "hstate_translator_ckpt.pt"
MAX_NEW_TOKENS = 30

PREFIXES = [
    "Seoul is the capital of",
    "Paris is the capital of",
]


@torch.no_grad()
def get_past_and_H_raw_excluding_last_token(model, input_ids: torch.Tensor):
    assert input_ids.size(1) >= 2
    prefix_excl_last = input_ids[:, :-1]
    H_raw, past, _ = extract_layer_inputs_gpt2(model, prefix_excl_last, attention_mask=None, use_cache=True)
    return H_raw, past


@torch.no_grad()
def greedy_generate_from_past(model, tokenizer, prefix_ids, past_excl_last, max_new_tokens: int):
    model.eval()
    input_ids = prefix_ids[:, -1:]
    generated = prefix_ids.clone()
    past = past_excl_last

    for _ in range(max_new_tokens):
        out = model(input_ids=input_ids, past_key_values=past, use_cache=True, return_dict=True)
        logits = out.logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)

        generated = torch.cat([generated, next_id], dim=1)
        past = out.past_key_values
        input_ids = next_id

    return tokenizer.decode(generated[0], skip_special_tokens=True)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    cfg = ckpt["config"]

    tokenizer = GPT2TokenizerFast.from_pretrained(cfg["MODEL_A_NAME"])
    tokenizer.pad_token = tokenizer.eos_token

    modelA = GPT2LMHeadModel.from_pretrained(cfg["MODEL_A_NAME"]).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(cfg["MODEL_B_NAME"]).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    translator = SharedSpaceHTranslator(
        a_layers=cfg["a_layers"], a_hidden=cfg["a_hidden"],
        b_layers=cfg["b_layers"], b_hidden=cfg["b_hidden"],
        q_dim=cfg["Q_DIM"], d_model=cfg["D_MODEL"], n_heads=cfg["N_HEADS"], dropout=cfg["DROPOUT"]
    ).to(device).eval()
    translator.load_state_dict(ckpt["translator_state"])

    specA = ModelKVSpec(
        n_layers=modelA.config.n_layer,
        n_heads=modelA.config.n_head,
        head_dim=modelA.config.n_embd // modelA.config.n_head,
        hidden_size=modelA.config.n_embd,
    )

    for prefix in PREFIXES:
        print("\n" + "=" * 80)
        print("PREFIX:", prefix)

        prefix_ids = tokenizer(prefix, return_tensors="pt").input_ids.to(device)
        if prefix_ids.size(1) < 2:
            print("prefix too short; skip")
            continue

        # originals on prefix[:-1]
        H_A_raw, pastA = get_past_and_H_raw_excluding_last_token(modelA, prefix_ids)
        H_B_raw, pastB = get_past_and_H_raw_excluding_last_token(modelB, prefix_ids)

        K_A, V_A = pack_past_key_values(pastA)

        # translate B -> A
        with torch.no_grad():
            H_B_to_A_raw = translator.translate_B_to_A(H_B_raw)

        K_B_to_A, V_B_to_A = recompute_kv_from_hidden_raw_gpt2(modelA, H_B_to_A_raw)

        print(
            f"[cosine] translated B->A vs A_original KV: "
            f"key={cosine_sim_flat(K_B_to_A, K_A):.6f}, "
            f"val={cosine_sim_flat(V_B_to_A, V_A):.6f}"
        )

        pastA_from_B = unpack_past_key_values(K_B_to_A, V_B_to_A, specA)

        out1 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA,         MAX_NEW_TOKENS)
        out2 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA_from_B,  MAX_NEW_TOKENS)
        out3 = greedy_generate_from_past(modelB, tokenizer, prefix_ids, pastB,         MAX_NEW_TOKENS)

        print("\n[generate] (1) A_original  -> model A")
        print(out1)
        print("\n[generate] (2) B_to_A      -> model A")
        print(out2)
        print("\n[generate] (3) B_original  -> model B")
        print(out3)


if __name__ == "__main__":
    main()

device: cuda

PREFIX: Seoul is the capital of
[cosine] translated B->A vs A_original KV: key=0.874151, val=0.572617

[generate] (1) A_original  -> model A
Seoul is the capital of South Korea, and is home to the country's largest and most famous tourist attraction.

The city is also home to the country's largest and

[generate] (2) B_to_A      -> model A
Seoul is the capital of































[generate] (3) B_original  -> model B
Seoul is the capital of South Korea, and is home to the country's largest concentration of foreign-born residents.

The city's population is estimated at around 1.

PREFIX: Paris is the capital of
[cosine] translated B->A vs A_original KV: key=0.852613, val=0.553735

[generate] (1) A_original  -> model A
Paris is the capital of the world's largest oil-rich state, and the world's second-largest oil producer.

The country's oil production is expected to reach

[generate] (2) B_to_A      -> model A
Paris is the capital of


























